# LLM Fine-Tuning on Google Colab

This notebook fine-tunes Mistral-7B using QLoRA on Google Colab's free T4 GPU.

## Runtime Settings

1. Go to **Runtime** → **Change runtime type**
2. Select **GPU** as Hardware accelerator
3. Choose **T4** or **V100** (available with Colab Pro)

## What This Notebook Does

- Loads Mistral-7B in 4-bit quantization (fits in 16GB VRAM)
- Applies LoRA adapters for efficient fine-tuning
- Trains on custom instruction dataset
- Saves adapter weights to Google Drive

**Estimated runtime:** 45-90 minutes on T4 GPU

**Estimated cost:** Free (Colab Free tier) or ~$0.50 (Colab Pro)

## Step 1: Environment Setup

Install all required packages.

In [ ]:
# Install PyTorch with CUDA
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# Install Hugging Face ecosystem
!pip install -q transformers peft trl datasets accelerate bitsandbytes

# Install experiment tracking (optional)
!pip install -q wandb

print("✅ All packages installed")

## Step 2: Mount Google Drive

This allows you to save model checkpoints and access datasets from Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Create directory for saving models
!mkdir -p /content/drive/MyDrive/llm-finetune/models
print("✅ Google Drive mounted")

## Step 3: Verify GPU

Check that we have a GPU and enough VRAM.

In [ ]:
import torch

print("=" * 50)
print("GPU Information")
print("=" * 50)
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"CUDA version: {torch.version.cuda}")
else:
    print("⚠️ No GPU detected. Go to Runtime → Change runtime type → GPU")

print("=" * 50)

## Step 4: Hugging Face Authentication (Optional)

Required for gated models like Llama-3 or Mistral.

In [ ]:
from huggingface_hub import login
from google.colab import userdata

# Option 1: Store token in Colab Secrets (recommended)
# Go to Secrets icon (key) in left panel → Add "HF_TOKEN"
try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
    print("✅ Logged in to Hugging Face")
except:
    print("⚠️ No HF_TOKEN found in Secrets. Skipping authentication.")
    print("   Add your token to Secrets for gated models.")

## Step 5: Load Model with QLoRA

We'll use 4-bit quantization to fit the 7B model in Colab's 16GB VRAM.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Model selection
MODEL_NAME = "mistralai/Mistral-7B-v0.1"  # or "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",           # Normal Float 4-bit
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,      # Double quantization for extra savings
)

print(f"Loading {MODEL_NAME}...")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

# Load model with quantization
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

# Prepare for k-bit training
model = prepare_model_for_kbit_training(model)

print("✅ Model loaded successfully")

## Step 6: Configure LoRA Adapters

In [ ]:
from peft import LoraConfig, get_peft_model

# LoRA configuration
lora_config = LoraConfig(
    r=8,                              # Rank: 8 (higher = more capacity)
    lora_alpha=32,                    # Alpha scaling factor
    target_modules=[                  # Modules to adapt
        "q_proj", "k_proj", 
        "v_proj", "o_proj",
    ],
    lora_dropout=0.05,                # Dropout for regularization
    bias="none",
    task_type="CAUSAL_LM",
)

# Apply LoRA to model
model = get_peft_model(model, lora_config)

# Print trainable parameters
model.print_trainable_parameters()

# Expected output:
# trainable params: 8,388,608 || all params: 7,241,732,160 || trainable%: 0.1158%

## Step 7: Prepare Dataset

For this example, we'll use a simple instruction dataset.

In [ ]:
from datasets import Dataset

# Sample training data (replace with your own)
training_data = [
    {
        "instruction": "Convert to formal language",
        "input": "hey can u send me the report asap",
        "output": "Hello, could you please send me the report at your earliest convenience?"
    },
    {
        "instruction": "Summarize in one sentence",
        "input": "The meeting was productive. We discussed Q3 results. Revenue increased by 15%. New products launched.",
        "output": "Q3 meeting highlighted 15% revenue growth and new product launches."
    },
    {
        "instruction": "Fix the grammar",
        "input": "The team have completed there project on time.",
        "output": "The team has completed their project on time."
    },
    # Add more examples for better results (100+)
]

# Convert to Hugging Face Dataset
dataset = Dataset.from_list(training_data)
print(f"Dataset size: {len(dataset)} examples")

# Format for instruction tuning
def format_example(example):
    return {
        "text": f"### Instruction:\n{example['instruction']}\n\n### Input:\n{example['input']}\n\n### Response:\n{example['output']}"
    }

dataset = dataset.map(format_example)
print(f"Example formatted text:\n{dataset[0]['text'][:200]}...")

## Step 8: Configure Training

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

# Training arguments
training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/llm-finetune/models/mistral-qlora",
    num_train_epochs=3,
    per_device_train_batch_size=2,      # Reduce if OOM
    gradient_accumulation_steps=8,       # Effective batch: 2×8=16
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    logging_steps=5,
    save_strategy="epoch",
    fp16=True,
    report_to="none",                    # Set to "wandb" for tracking
    max_grad_norm=1.0,
)

# Initialize trainer
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=512,
    tokenizer=tokenizer,
    packing=False,
)

print("✅ Trainer configured")

## Step 9: Start Training

This will take 45-90 minutes depending on GPU.

In [ ]:
print("🚀 Starting training...")
print(f"Dataset size: {len(dataset)}")
print(f"Estimated time: 45-90 minutes")
print("=" * 50)

# Train
trainer.train()

print("=" * 50)
print("✅ Training complete!")

## Step 10: Save Model

In [ ]:
# Save adapter weights
output_dir = "/content/drive/MyDrive/llm-finetune/models/mistral-qlora-final"
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)

print(f"✅ Model saved to: {output_dir}")

# Verify files
!ls -lh {output_dir}

## Step 11: Test Inference

In [ ]:
def generate_response(instruction, input_text=""):
    """Generate response using the fine-tuned model."""
    prompt = f"### Instruction:\n{instruction}\n\n### Input:\n{input_text}\n\n### Response:\n"
    
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )
    
    full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    response = full_text.split("### Response:\n")[-1].strip()
    return response

# Test examples
print("=" * 60)
print("Testing Fine-Tuned Model")
print("=" * 60)

test_cases = [
    ("Convert to formal language", "thx for ur help"),
    ("Fix the grammar", "she dont like apples"),
    ("Summarize", "Today was a great day. The sun was shining. Birds were singing. I felt happy."),
]

for instruction, input_text in test_cases:
    response = generate_response(instruction, input_text)
    print(f"\nInstruction: {instruction}")
    print(f"Input: {input_text}")
    print(f"Output: {response}")
    print("-" * 60)

## Step 12: Upload to Hugging Face Hub (Optional)

In [ ]:
# Push to Hugging Face Hub
# Uncomment and replace with your username

# model.push_to_hub("your-username/mistral-qlora-instruction")
# tokenizer.push_to_hub("your-username/mistral-qlora-instruction")

print("To push to Hub, uncomment the lines above and add your username.")

## Summary

✅ **Installed** all required packages  
✅ **Loaded** Mistral-7B with QLoRA (fits in 16GB VRAM)  
✅ **Trained** on custom instruction dataset  
✅ **Saved** model to Google Drive  
✅ **Tested** inference  

### Next Steps

1. Expand your dataset (100+ examples for production quality)
2. Try different models (Llama-3-8B, Phi-3-mini)
3. Experiment with LoRA hyperparameters
4. Deploy using vLLM or TGI

### Troubleshooting

| Issue | Solution |
|-------|----------|
| OOM error | Reduce `per_device_train_batch_size` to 1 |
| Slow training | Use `packing=True` in SFTTrainer |
| Poor results | Increase dataset size, train more epochs |
| GPU disconnected | Runtime → Disconnect and reconnect |